In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMerge\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadtest\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Platform.Utils.Geom;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
//Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;
Init();

In [3]:
//ExecutionQueues

In [4]:
//GetDefaultQueue()

In [5]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-Cylinder-Beam");

In [6]:
BoSSSshell.WorkflowMgm.Init("EE-Cylinder-Beam");

Project name is set to 'EE-Cylinder-Beam'.
Default Execution queue is chosen for the database.
Opening existing database 'C:\Users\miao\Documents\Database\EE-Cylinder-Beam'.


In [7]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

In [8]:
//wmg.DefaultDatabase

## Create grid

In [9]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xLeft = -0.3;
        double xRight = 2.5;
        double ySize = 0.4;
        int kelem = 20;

        double[] CutOut1Point1 = new double[2] {  0.15, 0.25 }; 
        double[] CutOut1Point2 = new double[2] {  0.25, 0.15 };
        
        //var CutOut1 = new BoSSS.Platform.Utils.Geom.BoundingBox(2);
        //CutOut1.AddPoint(CutOut1Point1);
        //CutOut1.AddPoint(CutOut1Point2);
        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, (int)((xRight - xLeft) * kelem) + 1);
        double[] Ynodes = GenericBlas.Linspace(0, ySize, (int)(ySize * kelem) + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall-upper");
        grd.EdgeTagNames.Add(3, "velocity_inlet_left");
        grd.EdgeTagNames.Add(4, "pressure_outlet_right");
        //grd.EdgeTagNames.Add(5, "wall_inside");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;

            if(Math.Abs(X[1] - 0) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - ySize) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [10]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocity(double[] X) {");
           stw.WriteLine("    return 1.5 * 1.0 * X[1] * (0.4 - X[1]) / (0.4 / 2).Pow2();");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return - ((X[0] - 0.2).Pow2() + (X[1] - 0.2).Pow2()) + 0.05 * 0.05;");
           //stw.WriteLine("    return -1.0;");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi2(double[] X) { ");
           //stw.WriteLine("    double h = 0.01;");
           //stw.WriteLine("    if(X[0] >= (0.25) && X[0] < (0.6 - h)) {");
           //stw.WriteLine("        return -(X[1] - 0.2).Pow2() + h * h;");
           //stw.WriteLine("    }");
           //stw.WriteLine("    if(X[0] < (0.25)) {");
           //stw.WriteLine("    return -((X[1] - 0.2).Pow2() + (X[0] - (0.25)).Pow2()) + h * h;");
           //stw.WriteLine("    }");
           //stw.WriteLine("    return -((X[1] - 0.2).Pow2() + (X[0] - (0.6 - h)).Pow2()) + h * h;");
           stw.WriteLine("    return -1.0;");
           stw.WriteLine("    }"); 

           stw.WriteLine(" static public double InitialVelocityVx(double[] X) { ");
           stw.WriteLine("    return 1.5 * 1.0 * X[1] * (0.4 - X[1]) / (0.4 / 2).Pow2() - 10 * Convert.ToInt32(((X[0]+0.1)*(X[0]+0.1)+(X[1]-0.2)*(X[1]-0.2))<=0.01) * (X[1]-0.2);");
           stw.WriteLine("    }"); 
            
           stw.WriteLine(" static public double InitialVelocityVy(double[] X) { ");
           stw.WriteLine("    return 0.0 + 10 * Convert.ToInt32(((X[0]+0.1)*(X[0]+0.1)+(X[1]-0.2)*(X[1]-0.2))<=0.01) * (X[0]+0.1);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocity(){
        return new Formula("BoundaryValues.InletVelocity", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi2(){
        return new Formula("BoundaryValues.InitialPhi2", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialVelocityVx(){
        return new Formula("BoundaryValues.InitialVelocityVx", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialVelocityVy(){
        return new Formula("BoundaryValues.InitialVelocityVy", AdditionalPrefixCode:GetPrefixCode());
    }
    
}

## Create Control file

In [11]:
public static ZLS_Control Channel( int p = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    //int D = 2;
    //AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Cylinder-Beam";
    C.SessionName = "Cylinder-Beam-Comsol-Cluster";
    //C.ProjectDescription = "Droplet running on pc";
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1e1;
    C.PhysicalParameters.rho_B = 1e1;
    C.PhysicalParameters.mu_A = 1e-2;
    C.PhysicalParameters.mu_B = 1e-2;
    //double sigma = 0.046;
    //C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        //Lame2 = 0.5e6,
        Lame2 = 1e3,
        PoissonsRatio = 0.5,
        Viscosity = 1e-2
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid());
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_InitialVelocityVx());
    C.AddInitialValue("VelocityX#B", BoundaryValueFactory.Get_InitialVelocityVx());
    C.AddInitialValue("VelocityY#A", BoundaryValueFactory.Get_InitialVelocityVy());
    C.AddInitialValue("VelocityY#B", BoundaryValueFactory.Get_InitialVelocityVy());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi2());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi2());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower", "VelocityX#A", BoundaryValueFactory.Get_Zero());
    C.AddBoundaryValue("wall_lower", "VelocityX#B", BoundaryValueFactory.Get_Zero());    
    C.AddBoundaryValue("wall-upper", "VelocityX#A", BoundaryValueFactory.Get_Zero());
    C.AddBoundaryValue("wall-upper", "VelocityX#B", BoundaryValueFactory.Get_Zero());
    C.AddBoundaryValue("velocity_inlet_left", "VelocityX#A", BoundaryValueFactory.Get_InletVelocity());
    C.AddBoundaryValue("velocity_inlet_left", "VelocityX#B", BoundaryValueFactory.Get_InletVelocity());
    C.AddBoundaryValue("pressure_outlet_right");
    //C.AddBoundaryValue("wall_inside");
    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    //C.PhysicalParameters.sliplength = 0.001;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    
    C.AdaptiveMeshRefinement = true;
    //double[] p11 = new double[2] {0.1, 0.1};
    //double[] p12 = new double[2] {0.3, 0.3};   
    //var ind1     = new BoSSS.Application.XNSE.AMRInBoundingBox(p11, p12);
    //ind1.maxRefinementLevel = AMRlvl - 1;
    //C.activeAMRlevelIndicators.Add(ind1);
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;

    //double[] p11 = new double[2] {0.0, 0.0};
    //double[] p12 = new double[2] {0.4, 0.4};   
    //var refineMesh = new BoundingBox(p11, p12);
    //var ind1     = new BoSSS.Solution.AMRLevelIndicatorLibrary.AMRInBoundingBox(refineMesh);
    //ind1.maxRefinementLevel = AMRlvl;
    //C.activeAMRlevelIndicators.Add(ind1);

    //C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRInBoundingBox(new BoundingBox(new double[,] { { 0.4, 0.4 }, { 0, 0 } })) { maxRefinementLevel = AMRlvl });

    #endregion

    C.DynamicLoadBalancing_On = false;
    C.DynamicLoadBalancing_Period = 50;
    C.DynamicLoadBalancing_RedistributeAtStartup = false;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    //C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
    C.Option_LevelSetEvolution2 = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;

    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    double dt = 1e-1;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = 1000;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [12]:
//double[] Density = new double[] {0.1, 1, 10, 100}; 
double[] Density = new double[] {1e1}; 

In [13]:
foreach(double Den in Density){
    var C_CTRLFILE = Channel(2, 0, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 4;
    JobLocal.Activate();
    JobLocal.ShowOutput(); 
}

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Cylinder-Beam-Comsol-Cluster ... 
Opening existing database '\\dc3\userspace\miao\cluster\EE-Cylinder-Beam'.
Set Database: { Session Count = 30; Grid Count = 1281; Path = C:\Users\miao\Documents\Database\EE-Cylinder-Beam }
Grid is not in database yet...
Grid successfully saved: 265a60b9-ea32-486f-b154-02430473d0ba


Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\EE-Cylinder-Beam-ZwoLevelSetSolver2024Mar21_103104.017477
copied 46 files.
   written file: control.obj
deployment finished.
Starting mini batch processor in external process...
Started mini batch processor on local machine, process id is 9236.
started.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [14]:
wmg.Sessions

#0: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:30:29	296dc0f7...
#1: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:26:59	fca695fc...
#2: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:21:45	42e6fb50...
#3: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:17:02	0c3f802f...
#4: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:15:46	7d9f26ec...
#5: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:13:38	ce7c73d1...
#6: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:12:14	07078d66...
#7: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:08:19	b267ba62...
#8: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:07:22	dae197fd...
#9: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 10:00:54	84657b3f...
#10: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 09:56:55	ad934eaa...
#11: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	03/21/2024 07:38:32	6d27e45d...
#1

In [15]:
//BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

In [16]:
//var outPath = wmg.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();